In [36]:
import os, glob, warnings, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import tensorflow as tf
import torch
import torch.nn.functional as F
from torchvision import transforms
import timm

print('TF     :', tf.__version__)
print('PyTorch:', torch.__version__)
print('timm   :', timm.__version__)

TF     : 2.19.0
PyTorch: 2.10.0+cu128
timm   : 1.0.26


In [37]:
# KONFIGURASI
CLASS_NAMES  = ['Kaca', 'Kardus', 'Kertas', 'Logam', 'Plastik', 'Residu']
IMG_SIZE     = 224
N_SAMPLE     = 300  

# Sesuaikan path model kamu
MODEL_A_PATH = '/kaggle/input/datasets/mieayam001/model-klasifikasi-sampah/Model_A.h5'
MODEL_B_PATH = '/kaggle/input/datasets/mieayam001/model-klasifikasi-sampah/Model_B.pth'

# Direktori test
TEST_DIR = '/kaggle/input/datasets/mieayam001/dataset-kalsifikasi/test'

In [38]:
# LOAD MODEL A
print('Loading Model A...')
model_a = tf.keras.models.load_model(MODEL_A_PATH)
print(f'  Input : {model_a.input_shape}')
print(f'  Output: {model_a.output_shape}')
print('Model A ✓')

Loading Model A...


  Input : (None, 224, 224, 3)
  Output: (None, 6)
Model A ✓


In [39]:
# LOAD MODEL B
print('Loading Model B...')

def load_model_b(path):
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(checkpoint, dict):
        for key in ['model_state_dict', 'state_dict']:
            if key in checkpoint:
                state_dict = checkpoint[key]
                break
        else:
            state_dict = checkpoint 
    else:
        return checkpoint.eval()

    # Deteksi prefix
    sample_key = next(iter(state_dict))
    has_model_prefix = sample_key.startswith('model.')
    print(f'  Contoh key: {sample_key}')

    if has_model_prefix:
        # Pakai wrapper agar prefix 'model.' cocok
        class TimmWrapper(torch.nn.Module):
            def __init__(self, num_classes):
                super().__init__()
                self.model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=num_classes)
            def forward(self, x):
                return self.model(x)

        wrapper = TimmWrapper(num_classes=len(CLASS_NAMES))
        wrapper.load_state_dict(state_dict, strict=True)
        print('  Load TimmWrapper (model.*) ✓')
        return wrapper.eval()
    else:
        base_model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(CLASS_NAMES))
        base_model.load_state_dict(state_dict, strict=True)
        print('  Load timm langsung ✓')
        return base_model.eval()

model_b = load_model_b(MODEL_B_PATH)
print('Model B ✓')

Loading Model B...
  Contoh key: model.conv_stem.weight
  Load TimmWrapper (model.*) ✓
Model B ✓


In [40]:
# FUNGSI PREPROCESSING & PREDIKSI 
def preprocess_keras(img_path):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, 0)

torch_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def preprocess_torch(img_path):
    img = Image.open(img_path).convert('RGB')
    return torch_transform(img).unsqueeze(0)

def predict_a(img_path):
    x = preprocess_keras(img_path)
    probs = model_a.predict(x, verbose=0)[0]
    idx = int(np.argmax(probs))
    return CLASS_NAMES[idx], round(float(probs[idx]) * 100, 2)

def predict_b(img_path):
    x = preprocess_torch(img_path)
    with torch.no_grad():
        probs = F.softmax(model_b(x), dim=1)[0].numpy()
    idx = int(np.argmax(probs))
    return CLASS_NAMES[idx], round(float(probs[idx]) * 100, 2)

print('Fungsi prediksi siap ✓')

Fungsi prediksi siap ✓


In [41]:
# KUMPULKAN 20 GAMBAR TEST 
all_images, all_labels = [], []

for cls in CLASS_NAMES:
    cls_dir = os.path.join(TEST_DIR, cls)
    if not os.path.exists(cls_dir):
        for d in os.listdir(TEST_DIR):
            if d.lower() == cls.lower():
                cls_dir = os.path.join(TEST_DIR, d)
                break
    imgs = (glob.glob(os.path.join(cls_dir, '*.jpg')) +
            glob.glob(os.path.join(cls_dir, '*.jpeg')) +
            glob.glob(os.path.join(cls_dir, '*.png')))
    all_images.extend(imgs)
    all_labels.extend([cls] * len(imgs))

print(f'Total gambar tersedia: {len(all_images)}')

random.seed(42)
per_class = max(1, N_SAMPLE // len(CLASS_NAMES))
sampled_images, sampled_labels = [], []

for cls in CLASS_NAMES:
    cls_imgs = [img for img, lbl in zip(all_images, all_labels) if lbl == cls]
    sample = random.sample(cls_imgs, min(per_class, len(cls_imgs)))
    sampled_images.extend(sample)
    sampled_labels.extend([cls] * len(sample))

remaining = [(img, lbl) for img, lbl in zip(all_images, all_labels) if img not in sampled_images]
random.shuffle(remaining)
for img, lbl in remaining:
    if len(sampled_images) >= N_SAMPLE:
        break
    sampled_images.append(img)
    sampled_labels.append(lbl)

print(f'Gambar yang ditest : {len(sampled_images)}')
print(pd.Series(sampled_labels).value_counts().to_string())

Total gambar tersedia: 1396
Gambar yang ditest : 300
Kaca       50
Kardus     50
Kertas     50
Logam      50
Plastik    50
Residu     50


In [42]:
# INFERENSI 
results = []

print(f"{'No':<4} {'File':<30} {'True':^10} {'Pred_A':^12} {'Conf_A':^8} {'Hasil_A':^9} {'Pred_B':^12} {'Conf_B':^8} {'Hasil_B':^9}")
print('-' * 110)

for i, (img_path, true_label) in enumerate(zip(sampled_images, sampled_labels)):
    img_name = Path(img_path).name

    pred_a, conf_a = predict_a(img_path)
    pred_b, conf_b = predict_b(img_path)

    a_correct = pred_a == true_label
    b_correct = pred_b == true_label

    hasil_a = 'Benar' if a_correct else 'Salah'
    hasil_b = 'Benar' if b_correct else 'Salah'

    results.append({
        'No'          : i + 1,
        'Image'       : img_name,
        'True_Label'  : true_label,
        'Pred_Model_A': pred_a,
        'Conf_A (%)'  : conf_a,
        'Hasil_A'     : hasil_a,
        'Pred_Model_B': pred_b,
        'Conf_B (%)'  : conf_b,
        'Hasil_B'     : hasil_b,
    })

    a_sym = '✓' if a_correct else '✗'
    b_sym = '✓' if b_correct else '✗'
    print(f"{i+1:<4} {img_name:<30} {true_label:^10} {pred_a:^12} {conf_a:>6.1f}%  {a_sym+' '+hasil_a:^9} {pred_b:^12} {conf_b:>6.1f}%  {b_sym+' '+hasil_b:^9}")

df = pd.DataFrame(results)
print(f'\nInferensi selesai  ({len(df)} gambar)')

No   File                              True       Pred_A     Conf_A   Hasil_A     Pred_B     Conf_B   Hasil_B 
--------------------------------------------------------------------------------------------------------------
1    R_5614.jpg                        Kaca        Kaca      100.0%   ✓ Benar      Kaca      100.0%   ✓ Benar 
2    R_5518.jpg                        Kaca        Kaca       82.0%   ✓ Benar      Kaca       99.9%   ✓ Benar 
3    R_5270.jpg                        Kaca        Kaca       94.7%   ✓ Benar      Kaca       98.7%   ✓ Benar 
4    R_5541.jpg                        Kaca        Kaca       90.8%   ✓ Benar      Kaca       99.9%   ✓ Benar 
5    R_5711.jpg                        Kaca        Kaca      100.0%   ✓ Benar      Kaca       97.8%   ✓ Benar 
6    R_5647.jpg                        Kaca        Kaca       52.2%   ✓ Benar      Kaca       61.1%   ✓ Benar 
7    R_5191.jpg                        Kaca        Kaca      100.0%   ✓ Benar      Kaca      100.0%   ✓ Benar 
8

In [43]:
# SIMPAN CSV 
output_path = '/kaggle/working/inferensi_hasil.csv'
df.to_csv(output_path, index=False)

print(f'CSV disimpan ke: {output_path}')
print(f'Ukuran file    : {os.path.getsize(output_path)/1024:.1f} KB')
print(f'Jumlah baris   : {len(df)}')
print()
print('Preview:')
df

CSV disimpan ke: /kaggle/working/inferensi_hasil.csv
Ukuran file    : 17.7 KB
Jumlah baris   : 300

Preview:


,No,Image,True_Label,Pred_Model_A,Conf_A (%),Hasil_A,Pred_Model_B,Conf_B (%),Hasil_B
0,1,R_5614.jpg,Kaca,Kaca,100.00,Benar,Kaca,100.00,Benar
1,2,R_5518.jpg,Kaca,Kaca,82.01,Benar,Kaca,99.92,Benar
2,3,R_5270.jpg,Kaca,Kaca,94.74,Benar,Kaca,98.66,Benar
3,4,R_5541.jpg,Kaca,Kaca,90.81,Benar,Kaca,99.91,Benar
4,5,R_5711.jpg,Kaca,Kaca,99.97,Benar,Kaca,97.80,Benar
...,...,...,...,...,...,...,...,...,...
295,296,residu_112.jpg,Residu,Residu,67.36,Benar,Residu,97.70,Benar
296,297,R_3514.jpg,Residu,Residu,99.84,Benar,Residu,100.00,Benar
297,298,netizen-geram-tetangganya-buang-sembarangan-pe...,Residu,Residu,57.98,Benar,Residu,99.96,Benar
298,299,R_3249.jpg,Residu,Residu,98.49,Benar,Residu,99.99,Benar
